# Silver 1: parse the PDFs and check each parse

Task `silver_parse` of the job `ingest_financebench`. Reads `bronze_pdfs` as a stream, parses each new PDF with `ai_parse_document` **once**, and checks the result.

| Result of the parse | Goes to |
| --- | --- |
| Complete: it has elements and no page errors | `silver_parsed` |
| Some pages failed (`Request rate limit exceeded`, `Internal error`) | `silver_log`, marked **retryable**. It is not stored: a document with missing pages would silently lose text |
| No elements: the PDF is over 500 pages, or the conversion failed | `silver_log`, marked **not retryable** |

The checks are in [checks.py](./checks.py) (plain Python, with tests in `tests/`). The tables are defined in [tables.py](./tables.py), and the parse-and-file function is in [parsing.py](./parsing.py). The retry of the failed parses is the next task, `retry_parses`.

**Using the stream's error handling.** A stream records finished micro-batches in its checkpoint. A failed batch does not move the checkpoint, so the next run restarts at that batch: nothing is lost and nothing is skipped. What a restarted batch must not do is parse the same PDFs again (that is billed), so each batch first drops the PDFs already in `silver_parsed` or `silver_log`. A restart then costs almost nothing.

**Challenge: `ai_parse_document` answers some pages with an error.** In the first full run 3,180 of 49,965 pages (6%), in 55 PDFs, failed with `Request rate limit exceeded` or `Internal error`, because 10 PDFs were parsed at once. **Solution:** check every parse, never store an incomplete one, parse two files at a time (`max_files_per_batch`), and retry the failures later.

In [ ]:
from tables import Names, create_tables
from parsing import parse_docs

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "finance_bench_rag")
dbutils.widgets.text("max_files_per_batch", "2")  # PDFs parsed at once. Small, so the parser's rate limit is not hit

n = Names(dbutils.widgets.get("catalog"), dbutils.widgets.get("schema"))
max_files_per_batch = int(dbutils.widgets.get("max_files_per_batch"))

create_tables(spark, n)

In [ ]:
def parse_new_pdfs(batch_df, batch_id):
    """One micro-batch: skip the PDFs that are already recorded (a restarted batch costs nothing), parse the rest."""
    known = spark.table(n.parsed).select("path").union(spark.table(n.log).filter("stage = 'parse'").select("path"))
    good, bad = parse_docs(spark, n, batch_df.join(known, "path", "left_anti"))
    print(f"batch {batch_id}: {good} parsed, {bad} logged as bad")


(
    spark.readStream.option("maxFilesPerTrigger", max_files_per_batch).table(n.bronze)
    .writeStream.foreachBatch(parse_new_pdfs)
    .option("checkpointLocation", n.parse_checkpoint)
    .trigger(availableNow=True)
    .start().awaitTermination()
)

In [ ]:
print("PDFs parsed (silver_parsed):", spark.table(n.parsed).count())
display(spark.table(n.failures).filter("stage = 'parse'").groupBy("retryable").count())